# Autoencoders
_Compressing MNIST, and putting the compression to work_

---

An autoencoder is a network trained to reproduce its own input. Written like that the task is
trivial, and the whole interest lies in the constraint placed between the input and the output:
the data must pass through a representation of much smaller dimension, the *latent space*. What
the network keeps in that bottleneck is what it needs in order to reconstruct, and that is what
we will look at.

This lab builds one on the [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database, makes
it convolutional, uses it to remove noise, and ends by asking what its latent space is good for.
Its variational counterpart is the subject of the **VAE lab**.

---

## Conventions used in this lab

This lab is written with `PyTorch`. Four conventions are used throughout and are worth
stating once and for all.

**1. Two input layouts, and the `view` that goes from one to the other.** A `DataLoader` built
on MNIST returns images as `(N, 1, 28, 28)`. The dense networks of the first sections take a
vector, so their inputs are flattened with `x.view(x.size(0), -1)` into `(N, 784)`; the
convolutional networks take the images as they come. Whenever a training or display function is
duplicated in this lab, that flattening is the only difference between the two versions.

**2. The decoders end with a sigmoid, and the loss is the binary cross-entropy.** Contrary to
the usual `PyTorch` practice of keeping raw scores inside the model, the decoders here output
values in $[0, 1]$, like the pixels of the normalized images. This follows the seminal article
[[Kingma & Welling, 2014]](https://arxiv.org/pdf/1312.6114.pdf) and keeps the reconstruction term
readable. The numerically stable alternative would be to output raw scores and use
`binary_cross_entropy_with_logits`; nothing in this lab depends on that choice.

**3. The test set plays the role of a validation set.** There is no third split here, and the
loss printed as `Val Loss` during training is computed on the test set. Nothing is ever
*selected* on it (no early stopping, no search over the hyperparameters), so the figures stay
honest; had we wanted to choose between several runs, a genuine validation set would have been
needed, as in the VisionCNN lab.

**4. Everything is sent to the same `device`.** The models and *all* the tensors given to them.
A tensor left on the CPU while the model sits on the GPU is the most common error in this lab,
and it does not show up on a machine that has no GPU.

## Setting up the environment

In [ ]:
import math
import random

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset, ConcatDataset
from torchvision import datasets
from torchvision.transforms import v2
from torchinfo import summary

print("torch version:", torch.__version__)

In [ ]:
from tqdm import tqdm
#from tqdm.notebook import tqdm

from sklearn.manifold import TSNE
from scipy.stats import norm

In [ ]:
# All the tensors and models of this lab will be sent to this device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Pinning host memory only speeds up transfers towards an accelerator: without one,
# it does nothing, and recent versions of PyTorch warn about it at every DataLoader.
PIN_MEMORY = (device.type == "cuda")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## The [MNIST](https://en.wikipedia.org/wiki/MNIST_database) database

As seen earlier in the course, the stability of the algorithms is improved by normalizing the data.
In `PyTorch`, transformations are applied to the data at loading time: this is what we do here.

In [ ]:
# Transform: from a PIL image to a float tensor with values in [0, 1]
transform = v2.Compose([
    v2.ToImage(),                           # PIL image -> tensor, with an explicit channel dimension
    v2.ToDtype(torch.float32, scale=True),  # uint8 in [0, 255] -> float32 in [0, 1]
])

# Load datasets
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [ ]:
print(f"Train set: {len(train_dataset)} images of size {train_dataset.data.shape[1]} x {train_dataset.data.shape[2]}")
print(f"Test set:  {len(test_dataset)} images")

In order to train the networks more easily afterwards, we create a `DataLoader` to access the
data. In particular, we need to specify the size of the (future) training batches.

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 2,
    pin_memory = PIN_MEMORY)

test_loader = DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 2,
    pin_memory = PIN_MEMORY)

A batch can then be accessed with the command `next(iter(train_loader))`.

In [ ]:
images, labels = next(iter(train_loader))

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

The following code displays sample images.

In [ ]:
n = 10

plt.figure(figsize=(20, 4))
for i in range(n):
    ax = plt.subplot(2, n, i+1)
    plt.imshow(images[i][0], cmap="gray")
    ax.grid(False)
    plt.axis("off")
plt.show()

## A first very simple autoencoder

First, we build a very simple architecture where:

* The **encoder** is a dense layer of 32 neurons (the dimension of the latent variable) with a
  $\texttt{ReLU}$ activation function:
$$ \texttt{ReLU}(x) = \max(0, x) \,; $$

* The **decoder** is a dense layer of $784 = 28\times28$ neurons (the dimension of the input)
  with a sigmoid activation function:
$$ \sigma(x) = \frac{1}{1+\mathrm{e}^{-x}} \,. $$

##### <i style="color:teal">**Todo:** Write the simple model described above</i>

In [ ]:
### TO BE COMPLETED ###

n_input = 784
n_latent = 32

In [ ]:
### TO BE COMPLETED ###

class SimpleAutoencoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(n_input, n_latent),
            nn.ReLU()
        )
        # Decoder
        self.decoder = ... ### TO BE COMPLETED ###

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


# Instantiate model
simple_autoencoder = SimpleAutoencoder(n_input, n_latent)
summary(simple_autoencoder, input_size=(1, n_input))

In [ ]:
# %load solutions/ae/SimpleAutoencoder.py

We can now train the model. Note that _the target variable is the original image._

Two points deserve attention, and they will come back at every training loop of this lab.

`model.train()` and `model.eval()` do not train or evaluate anything: they switch the *mode* of
the layers whose behaviour differs between the two. No such layer appears in this first model,
but the switch is written from the start because it will matter as soon as a `nn.Dropout` or a
`nn.BatchNorm` shows up.

Gradients accumulate in `PyTorch`: `optimizer.zero_grad()` resets them before each `backward()`.
Omitting it silently sums the gradients of successive batches. And `.item()` extracts a plain
Python number out of a one-element tensor; accumulating the tensors themselves would keep the
whole computational graph alive, and the memory with it.

In [ ]:
n_input = 784
n_latent = 32

EPOCHS = 10  #50
LEARNING_RATE = 1e-3

In [ ]:
simple_autoencoder = SimpleAutoencoder(n_input, n_latent)
simple_autoencoder.to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(simple_autoencoder.parameters(), lr=LEARNING_RATE)

# --- #
# Training loop
for epoch in range(EPOCHS):
    simple_autoencoder.train()
    train_loss = 0

    for inputs, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        # Move data to the same device as model
        inputs = inputs.to(device)
        inputs_flat = inputs.view(inputs.size(0), -1)

        optimizer.zero_grad()
        outputs = simple_autoencoder(inputs_flat)
        loss = criterion(outputs, inputs_flat)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    simple_autoencoder.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            inputs_flat = inputs.view(inputs.size(0), -1)
            outputs = simple_autoencoder(inputs_flat)
            loss = criterion(outputs, inputs_flat)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(test_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

**Note**: here we used the binary cross-entropy as the loss function, following the seminal
article [[Kingma & Welling, 2014]](https://arxiv.org/pdf/1312.6114.pdf). This choice is
legitimate because of the _sigmoid_ activation function, which brings the output of the decoder
back into $[0, 1]$. See convention 2 at the top of the notebook.

##### <i style="color:teal">**Todo:** Check the performance of the model by displaying sample test images and their reconstruction</i>

To do this, write a `plot_images` function that displays the images contained in the
$\texttt{imgs}$ list.

* $\texttt{imgs}$ is a list whose items are themselves lists (or batches) of images;
* each of the images of a same item must be displayed on the same row of the global figure;
* in the end, the global figure has as many columns as the size of each item of $\texttt{imgs}$
  (they all have the same size, namely $n$), and as many rows as the size of $\texttt{imgs}$.

> This function is used by the global `visualize_autoencoder` function defined immediately
> afterwards, which relies on a series of auxiliary functions to:
> * check that the autoencoder being "visualized" sits on the right device, with `check_device_model`;
> * draw $n$ images at random from the test dataset, with `select_n_samples`;
> * display these images, their encoded and their decoded counterparts, with the `plot_images`
>   function you are about to write.
>
> You can freely use `check_device_model` and `select_n_samples` later on.

In [ ]:
### TO BE COMPLETED ###

def plot_images(imgs, sz, titles, n, cmap="gray"):
    """Display several rows of images, one row per entry of `imgs`."""
    num_rows = len(imgs)
    plt.figure(figsize=(2*n, 2*num_rows))

    for row, images in enumerate(imgs):
        # Accept tensors as well as arrays: bring everything back to numpy on the CPU
        if torch.is_tensor(images):
            images = images.detach().cpu().numpy()

        [...] ### TO BE COMPLETED ###

        plt.subplot(num_rows, n, row*n + 1).set_title(titles[row], fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
# %load solutions/ae/plot_images.py

The three functions below are given. `check_device_model` and `select_n_samples` are used
everywhere in the lab; `visualize_autoencoder` is the model to follow for the display functions
you will be asked to write later.

In [ ]:
def check_device_model(model, device=device):
    """Move `model` to `device`. Calling it on a model already there costs nothing."""
    return model.to(device)


# --- #

def select_n_samples(loader, n, device=device):
    """Draw n images at random from the dataset of `loader`, as a [n, 1, 28, 28] tensor."""
    dataset = loader.dataset
    indices = torch.randperm(len(dataset))[:n]
    inputs_list = [dataset[i][0].unsqueeze(0) for i in indices]
    inputs = torch.cat(inputs_list, dim=0).to(device)
    return inputs


# --- #

def visualize_autoencoder(autoencoder,
                          loader = test_loader,
                          input_sz = (28, 28),
                          latent_sz = (4, 8),
                          n = 10,
                          device = device
                         ):
    """Display n test images, their latent representation and their reconstruction.

    The latent space has 32 dimensions here, laid out as a 4x8 image so that it can be
    shown at all: there is nothing spatial about that layout.
    """

    autoencoder = check_device_model(autoencoder, device=device)
    autoencoder.eval()

    inputs = select_n_samples(loader=loader, n=n, device=device)  # [n, 1, 28, 28]
    inputs_flat = inputs.view(inputs.size(0), -1)                 # flatten for encoder/decoder

    with torch.no_grad():
        encoded = autoencoder.encoder(inputs_flat)
        decoded = autoencoder.decoder(encoded)

    plot_images(
        imgs = [inputs, encoded, decoded],
        sz = [input_sz, latent_sz, input_sz],
        titles = ['Original', 'Encoded', 'Decoded'],
        n = n
    )

In [ ]:
visualize_autoencoder(simple_autoencoder)

##### <i style="color:teal">**Todo:** Check that decoding a latent representation gives the same thing as encoding and decoding the original data</i>

Write a function displaying, on three different rows and for about ten images:

* on the 1st row, the original image;
* on the 2nd row, the "auto-encoded" image, _i.e._ the result of the full forward pass through
  the autoencoder;
* on the 3rd row, the image encoded by the encoder, then decoded by the decoder.

Use the `visualize_autoencoder` function defined above as a model, in particular for the syntax.

In [ ]:
### TO BE COMPLETED ###

def check_encode_decode(autoencoder,
                        loader = test_loader,
                        input_sz = (28, 28),
                        n = 10,
                        device = device
                       ):
    """Check that encoding then decoding gives the same thing as the full forward pass."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/ae/check_encode_decode.py

In [ ]:
check_encode_decode(simple_autoencoder)

## A more realistic autoencoder

We now make the architecture _slightly_ more complex: one hidden layer of 128 neurons on each
side, and the encoder and the decoder written as two separate modules.

In [ ]:
# Encoder
class Encoder(nn.Module):
    """Dense encoder: 784 -> 128 -> n_latent, with a ReLU on the hidden layer only."""

    def __init__(self, n_input, n_latent):
        super().__init__()
        self.fc1 = nn.Linear(n_input, 128)
        self.fc2 = nn.Linear(128, n_latent)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# Decoder
class Decoder(nn.Module):
    """Dense decoder: n_latent -> 128 -> 784, with a ReLU on the hidden layer only."""

    def __init__(self, n_latent, n_output):
        super().__init__()
        self.fc1 = nn.Linear(n_latent, 128)
        self.fc2 = nn.Linear(128, n_output)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, n_input, n_latent):
        super().__init__()
        self.encoder = Encoder(n_input, n_latent)
        self.decoder = Decoder(n_latent, n_input)

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


# Instantiate models
encoder = Encoder(n_input, n_latent)
decoder = Decoder(n_latent, n_input)
autoencoder = Autoencoder(n_input, n_latent)

# Print summaries
print("Autoencoder")
summary(autoencoder, input_size=(1, n_input))
print("\n Encoder")
summary(encoder, input_size=(1, n_input))
print("\n Decoder")
summary(decoder, input_size=(1, n_latent))

The training is performed with a loop similar to the one defined for `simple_autoencoder`.

In [ ]:
EPOCHS = 10  #50
LEARNING_RATE = 1e-3

autoencoder = Autoencoder(n_input, n_latent)
autoencoder.to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=LEARNING_RATE)

# --- #
# Training loop
for epoch in range(EPOCHS):
    autoencoder.train()
    train_loss = 0

    for inputs, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        inputs = inputs.to(device)
        inputs_flat = inputs.view(inputs.size(0), -1)

        optimizer.zero_grad()
        outputs = autoencoder(inputs_flat)
        loss = criterion(outputs, inputs_flat)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    autoencoder.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            inputs_flat = inputs.view(inputs.size(0), -1)
            outputs = autoencoder(inputs_flat)
            loss = criterion(outputs, inputs_flat)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(test_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# --- #
visualize_autoencoder(autoencoder)

We increased the complexity of the model, and yet the results are less convincing.

##### <i style="color:teal">**Question:** Why do you think this is? Taking the first model as a guide, suggest an improvement that would give better results.</i>

**[Solution]**

<!--
The decoder no longer ends with a sigmoid: its last layer is a plain `nn.Linear`, whose output
runs over the whole real line, while the target pixels live in [0, 1]. The network spends part
of its capacity learning to stay inside that interval, which the sigmoid gave it for free.

The loss changed too, from binary cross-entropy to mean square error, and the two go together:
the BCE is only defined for outputs in [0, 1], so it could not have been kept as is.

The fix is to put the sigmoid back on the output layer, and with it the binary cross-entropy.
Note that the extra depth is not the problem; it is not the reason the reconstruction is worse.
-->

##### <i style="color:teal">**Todo:** Implement that improvement</i>

Take the opportunity to define a `train_autoencoder` function, so that other autoencoders can be
trained easily later on.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/Autoencoder_improved.py

### Influence of the dimension of the latent space

With the dimension of the latent space we used, a (relatively) low reconstruction error is
observed. We would like to study the influence of that dimension on the reconstruction error.

##### <i style="color:teal">**Todo:** Draw a _curve_ (with a handful of points only) showing the reconstruction error as a function of the dimension of the latent space</i>

What is the smallest dimension of the latent space that still allows a reasonable reconstruction
of the data, with the network provided?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/latent_dimension.py

In the previous example, the autoencoder is constrained only by the size of the hidden layer.
Other ways of constraining that space could have been considered, for instance by introducing
sparsity constraints (sparse autoencoders). We do not address this here.

## Convolutional autoencoders

In the previous sections, we saw very simple autoencoders whose encoder and decoder parts are
perceptrons. As seen during the course, both can be made of more layers, and of layers of a
different kind. Convolutional layers, in particular, are the layers to use when dealing with
images.

From here on the images are no longer flattened: they enter the network as `(N, 1, 28, 28)`.

##### <i style="color:teal">**Todo:** Implement a convolutional autoencoder with the following architecture</i>

**Encoder:**
* two convolution layers, 16 filters of size 3x3;
* a max-pooling layer with 2x2 filters;
* two convolution layers, 8 filters of size 3x3;
* a max-pooling layer with 2x2 filters.

**Decoder:**
* two convolution layers, 8 filters of size 3x3;
* an upsampling layer with 2x2 filters;
* two convolution layers, 16 filters of size 3x3;
* an upsampling layer with 2x2 filters;
* a convolution layer, 1 filter of size 3x3, with a $\texttt{sigmoid}$ activation.

All paddings are $\texttt{same}$ and all activation functions of the convolution layers, except
the last one, are $\texttt{ReLU}$.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/ConvAutoencoder.py

##### <i style="color:teal">**Todo:** Write the counterpart of the `train_autoencoder` function, adapted to the convolutional autoencoder defined above</i>

Pay particular attention to the size of the inputs of the network.

In [ ]:
### TO BE COMPLETED ###

def train_convolutional_autoencoder(autoencoder,
                                    train_loader = train_loader,
                                    test_loader = test_loader,
                                    criterion = nn.BCELoss(),
                                    EPOCHS = 10,  #50
                                    LEARNING_RATE = 1e-3,
                                    device = device,
                                    use_tqdm = True
                                   ):
    """Same loop as `train_autoencoder`, without the flattening."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/ae/train_convolutional_autoencoder.py

##### <i style="color:teal">**Todo:** Write the counterpart of the `visualize_autoencoder` function</i>

The latent representation is now a $[8, 7, 7]$ volume, _i.e._ 392 values: the `latent_sz`
argument lays them out as a $14\times28$ image so that they can be displayed.

In [ ]:
### TO BE COMPLETED ###

def visualize_convolutional_autoencoder(autoencoder,
                                        loader = test_loader,
                                        input_sz = (28, 28),
                                        latent_sz = (14, 28),
                                        n = 10,
                                        device = device
                                       ):
    """Display n test images, their latent representation and their reconstruction."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/ae/visualize_convolutional_autoencoder.py

In [ ]:
convolutional_autoencoder = ConvAutoencoder()
convolutional_autoencoder, _, _ = train_convolutional_autoencoder(convolutional_autoencoder, use_tqdm=True)
visualize_convolutional_autoencoder(convolutional_autoencoder)

## Application to denoising

We now know how to build a convolutional autoencoder. Let us see how to use it to solve an image
denoising problem. First, we create fake noisy data out of the MNIST data.

The dataset below returns a **triple** `(noisy image, clean image, label)`, and not a pair: the
training function of this section is the only one of the lab that receives an input and a target
that differ, and that is exactly what makes the network learn something other than the identity.

In [ ]:
class NoisyMNIST(Dataset):
    """MNIST with additive Gaussian noise, clipped back to [0, 1].

    The noise is drawn in `__getitem__`, so it changes at every epoch: the network never
    sees the same corrupted image twice, which acts as data augmentation.
    """

    def __init__(self, original_dataset, noise_factor=0.4):
        self.dataset = original_dataset
        self.noise_factor = noise_factor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        noisy_img = img + self.noise_factor * torch.randn_like(img)
        noisy_img = torch.clamp(noisy_img, 0., 1.)
        return noisy_img, img, label

In [ ]:
train_noisy_dataset = NoisyMNIST(train_dataset)
test_noisy_dataset  = NoisyMNIST(test_dataset)

# --- #

train_noisy_loader = DataLoader(
    train_noisy_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 0,
    pin_memory = PIN_MEMORY
)

test_noisy_loader = DataLoader(
    test_noisy_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 0,
    pin_memory = PIN_MEMORY
)

Let us take a look at the noise we created.

##### <i style="color:teal">**Todo:** Display a few images and their noisy counterparts</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/ae/noisy_images.py

##### <i style="color:teal">**Todo:** Adapt the `train_convolutional_autoencoder` function to the dataset induced by the `NoisyMNIST` class</i>

Two things change: the loaders return a triple, and the target of the loss is the clean image.

In [ ]:
### TO BE COMPLETED ###

def train_noisy_convolutional_autoencoder(autoencoder,
                                          train_loader = train_noisy_loader,
                                          test_loader = test_noisy_loader,
                                          criterion = nn.BCELoss(),
                                          EPOCHS = 10,  #50
                                          LEARNING_RATE = 1e-3,
                                          device = device,
                                          use_tqdm = True
                                         ):
    """Train the autoencoder to go from the noisy image to the clean one."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/ae/train_noisy_convolutional_autoencoder.py

##### <i style="color:teal">**Todo:** Write the display function matching this dataset</i>

Three rows are wanted: the clean image, the noisy image given to the network, and what the
network makes of it. `visualize_convolutional_autoencoder` cannot be reused as is here: fed with
`test_noisy_loader`, it would take the noisy image for the original and never show the clean one.

In [ ]:
### TO BE COMPLETED ###

def visualize_noisy_convolutional_autoencoder(autoencoder,
                                              loader = test_noisy_loader,
                                              input_sz = (28, 28),
                                              n = 10,
                                              device = device
                                             ):
    """Display n test images, their noisy version, and what the autoencoder makes of it."""

    [...] ### TO BE COMPLETED ###

In [ ]:
# %load solutions/ae/visualize_noisy_convolutional_autoencoder.py

In [ ]:
denoising_autoencoder = ConvAutoencoder()
denoising_autoencoder, _, _ = train_noisy_convolutional_autoencoder(denoising_autoencoder, use_tqdm=True)
visualize_noisy_convolutional_autoencoder(denoising_autoencoder)

## Semi-supervised learning: classification <small style="color:orangered">(to go further)</small>

The idea of semi-supervised learning in this context is to take advantage of the ability of the
latent space to represent the data well, or so we hope.

Specifically, we have a small labeled dataset and a large unlabeled one. We compare two
classifiers: one trained on the original data, the other on the encoded data. If the latent
space were ideally built, the second should clearly outperform the first. It does not: the two
land within a few points of each other. On five hundred test images one point is five images,
and the standard error on an accuracy near $0.85$ is about one and a half points, so a gap of
that size is not something to conclude from. The comparison is the point here, not the winner.

The first step is to create small subsets of labeled data. We keep $n=100$ images per digit for
training and 50 per digit for testing.

> The subsets are built from `train_dataset.targets`, the vector of the labels, and not by
> iterating over the dataset itself: going through the 60 000 images ten times would apply the
> `transform` 600 000 times, for several minutes of waiting and the very same result.

In [ ]:
convolutional_autoencoder = ConvAutoencoder()
convolutional_autoencoder, _, _ = train_convolutional_autoencoder(convolutional_autoencoder, use_tqdm=True)

In [ ]:
nb_train = 100
nb_test = 50

train_targets = train_dataset.targets
test_targets  = test_dataset.targets

train_indices = []
test_indices  = []

for i in range(10):
    train_indices.extend(torch.nonzero(train_targets == i, as_tuple=True)[0][:nb_train].tolist())
    test_indices.extend(torch.nonzero(test_targets == i, as_tuple=True)[0][:nb_test].tolist())

train_indices = torch.tensor(train_indices)[torch.randperm(len(train_indices))]
test_indices  = torch.tensor(test_indices)[torch.randperm(len(test_indices))]

few_train_dataset = Subset(train_dataset, train_indices)
few_test_dataset  = Subset(test_dataset, test_indices)

print(f"{len(few_train_dataset)} labeled training images, {len(few_test_dataset)} test images")

We then encode this subset of data, using the convolutional autoencoder trained on all the data.

In [ ]:
def encode_dataset(autoencoder, dataset, batch_size=64, device=device):
    """Encode a whole dataset and return the latent representations and their labels."""
    autoencoder = check_device_model(autoencoder, device=device)
    autoencoder.eval()

    encoded_list, labels_list = [], []
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            encoded = autoencoder.encoder(inputs)
            encoded_list.append(encoded.cpu())
            labels_list.append(labels)

    return torch.cat(encoded_list), torch.cat(labels_list)


few_encoded_train, few_labels_train = encode_dataset(convolutional_autoencoder, few_train_dataset)
few_encoded_test,  few_labels_test  = encode_dataset(convolutional_autoencoder, few_test_dataset)

We can then define, and train, a classifier on the latent space. It is a single linear layer, on
purpose: what is being compared is the quality of the two representations, not the capacity of
the two classifiers.

In [ ]:
class LatentClassifier(nn.Module):
    """A single linear layer, from the representation to the ten classes."""

    def __init__(self, latent_dim, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(latent_dim, n_classes)

    def forward(self, x):
        return self.fc(x)


def train_classifier(classifier, dataset, EPOCHS=30, BATCH_SIZE=20, verbose=False):
    """Train a classifier with a cross-entropy loss. `classifier` outputs raw scores."""
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    classifier.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = classifier(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.shape[0]
        total_loss /= len(loader.dataset)
        if verbose or epoch == EPOCHS - 1:
            print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")
    return classifier


few_encoded_train_flat = few_encoded_train.view(few_encoded_train.size(0), -1)
few_encoded_test_flat  = few_encoded_test.view(few_encoded_test.size(0), -1)
n_latent_conv = few_encoded_train_flat.shape[1]

latent_classifier = LatentClassifier(latent_dim=n_latent_conv).to(device)
latent_dataset    = TensorDataset(few_encoded_train_flat, few_labels_train)
latent_classifier = train_classifier(latent_classifier, latent_dataset)

In [ ]:
def evaluate_classifier(classifier, dataset):
    """Accuracy of `classifier` over `dataset`."""
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    classifier.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = classifier(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += outputs.shape[0]
    return correct / total


latent_acc = evaluate_classifier(latent_classifier, TensorDataset(few_encoded_test_flat, few_labels_test))
print(f"Accuracy on latent test set: {latent_acc:.4f}")

Finally, we define and train a classifier on the original space.

In [ ]:
class ImageClassifier(nn.Module):
    """The same single linear layer, applied to the raw pixels."""

    def __init__(self, input_dim, n_classes=10):
        super().__init__()
        self.fc = nn.Linear(input_dim, n_classes)

    def forward(self, x):
        return self.fc(x)


def flatten_dataset(dataset):
    """Return the whole dataset as one flattened tensor of images and one of labels."""
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    for inputs, labels in loader:
        return inputs.view(inputs.shape[0], -1), labels


inputs_few_train, labels_few_train = flatten_dataset(few_train_dataset)
inputs_few_test,  labels_few_test  = flatten_dataset(few_test_dataset)

image_classifier = ImageClassifier(inputs_few_train.shape[1]).to(device)
image_dataset    = TensorDataset(inputs_few_train, labels_few_train)
image_classifier = train_classifier(image_classifier, image_dataset)

image_acc = evaluate_classifier(image_classifier, TensorDataset(inputs_few_test, labels_few_test))
print(f"\n Accuracy on image test set: {image_acc:.4f}")

##### <i style="color:teal">**Question:** Why does the latent representation bring no clear gain over the raw pixels here?</i>

**[Solution]**

<!--
The two classifiers land within a few points of each other, the latent one slightly ahead on the
runs made while writing this lab, by one to three points depending on the run. On five hundred
test images the standard error on an accuracy near $0.85$ is about one and a half points, so a
gap of that size sits at the edge of what this test set can resolve. There may be a small real
advantage there; what there is not is the clear gain one might have hoped for, and that absence
is what is worth explaining.

Nothing in the training of the autoencoder asks the latent space to separate the classes. It is
trained to reconstruct, so it keeps whatever reconstruction needs (stroke thickness, slant,
position), which is not the same thing as what is needed to tell a 3 from an 8.

The comparison is also stacked in a way that leaves little room. The representation of the
convolutional autoencoder has 392 dimensions, half the 784 of the image, so there is hardly any
compression to take advantage of. And MNIST is close enough to linearly separable that a single
linear layer on the raw pixels already does well.

Two things would make a difference visible: a much smaller latent space, and a loss that asks
the encoder for something beyond reconstruction.
-->

---
## Where this leads

The latent space of this autoencoder is a set of points, and nothing more. The network was never
asked to make it look like anything in particular: nothing says what lies *between* two codes,
and nothing says where a code should be drawn from if we wanted the decoder to invent an image
rather than reproduce one. Everything this lab did with the latent space went *through* an
image: encode this one, decode that one.

Nothing stops us from drawing a vector at random and decoding it, which is one line of code and
worth trying before reading on. What comes out is not a digit. The reason is the one just
stated: nothing ever constrained *where* the codes of the real images live, so a vector drawn
from nowhere in particular lands nowhere in particular.

That is the honest state of things at the end of this lab, and it is the question the rest of
the course answers, twice and in two opposite ways.

**The GAN lab drops the encoder.** There is no code to read an image into, because there is
nothing to encode: a generator is trained to turn noise straight into images, judged by a second
network whose only job is to tell real from generated. Sharp images, and no way to ask the model
what it thinks of an image it is shown.

**The VAE lab keeps the encoder** and replaces the set of points by a probability distribution.
The encoder and the decoder barely change; one term is added to the loss. What comes with it is
generation, a latent space one can walk in, and a reconstruction error that means something.